In [ ]:
# Import the necessary libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load the grayscale image
grayscale_image = cv2.imread('pic5.jpg', cv2.IMREAD_GRAYSCALE)

# Check if the image was loaded successfully
if grayscale_image is None:
    print("Error: Could not load image.")
else:
    # Convert the grayscale image to RGB
    rgb_image = cv2.cvtColor(grayscale_image, cv2.COLOR_GRAY2RGB)

    # Display the original grayscale image
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.title('Grayscale Image')
    plt.imshow(grayscale_image, cmap='gray')
    plt.axis('off')

    # Display the converted RGB image
    plt.subplot(1, 2, 2)
    plt.title('RGB Image')
    plt.imshow(rgb_image)
    plt.axis('off')

    plt.show()

    # Check if the converted image is indeed RGB
    if rgb_image.ndim == 3 and rgb_image.shape[2] == 3:
        print("The converted image is RGB.")
    else:
        print("The converted image is not RGB.")

In [ ]:
colored_image = cv2.applyColorMap(grayscale_image, cv2.COLORMAP_AUTUMN)
colored_image_rgb = cv2.cvtColor(colored_image, cv2.COLOR_BGR2RGB)

In [ ]:
plt.subplot(1, 3, 3)
plt.title('Colorized Image')
plt.imshow(colored_image_rgb)
plt.axis('off')

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)  # grayscale
plt.subplot(1, 3, 2)  # rgb
plt.subplot(1, 3, 3)  # colorized

In [ ]:
plt.imshow(colored_image_rgb)


In [ ]:
import torch
from colorizers import eccv16

# Load the pretrained colorization model.
# This is a PyTorch port of the exact same "colorization_release_v2" weights used
# above, published by the model's author (richzhang/colorization). We use this
# instead of cv2.dnn.readNet(...) because OpenCV 5 removed its Caffe importer,
# and the original .caffemodel/.prototxt pair can no longer be loaded directly.
colorizer = eccv16(pretrained=True).eval()

# Prepare image (same L-channel pipeline as the original Caffe-based approach)
img = grayscale_image.astype(np.float32) / 255.0
img_lab = cv2.cvtColor(cv2.merge([img, img, img]), cv2.COLOR_RGB2Lab)
L = img_lab[:, :, 0]

L_tensor = torch.from_numpy(L)[None, None, :, :].float()

# Run colorization
with torch.no_grad():
    ab_tensor = colorizer(L_tensor)
ab = ab_tensor[0].permute(1, 2, 0).numpy()
ab = cv2.resize(ab, (grayscale_image.shape[1], grayscale_image.shape[0]))

# Combine L + ab and convert to RGB
result_lab = np.concatenate([img_lab[:, :, 0:1], ab], axis=2)
result_bgr = cv2.cvtColor(result_lab, cv2.COLOR_Lab2BGR)
result_rgb = (np.clip(result_bgr, 0, 1) * 255).astype(np.uint8)
result_rgb = cv2.cvtColor(result_rgb, cv2.COLOR_BGR2RGB)

# Display
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title('Grayscale')
plt.imshow(grayscale_image, cmap='gray')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title('AI Colorized')
plt.imshow(result_rgb)
plt.axis('off')
plt.show()
